In [9]:
import tensorflow as tf
tf.compat.v1.disable_v2_behavior()  # 启用 TensorFlow 1.x 兼容模式
import numpy as np

# ==================== 数据加载 ====================
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 归一化并 reshape
x_train = x_train.reshape(-1, 784) / 255.0
x_test = x_test.reshape(-1, 784) / 255.0

# 标签 one-hot 编码
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# 包装器，模拟旧版 mnist 对象
class DataSet:
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels
        self._index_in_epoch = 0
        self._epochs_completed = 0
        self._num_examples = images.shape[0]

    def next_batch(self, batch_size):
        start = self._index_in_epoch
        if start + batch_size > self._num_examples:
            perm = np.arange(self._num_examples)
            np.random.shuffle(perm)
            self.images = self.images[perm]
            self.labels = self.labels[perm]
            self._epochs_completed += 1
            start = 0
            self._index_in_epoch = batch_size
            return self.images[start:self._index_in_epoch], self.labels[start:self._index_in_epoch]
        else:
            self._index_in_epoch += batch_size
            end = self._index_in_epoch
            return self.images[start:end], self.labels[start:end]

class MNISTWrapper:
    def __init__(self, train_images, train_labels, test_images, test_labels):
        self.train = DataSet(train_images, train_labels)
        self.test = DataSet(test_images, test_labels)

mnist = MNISTWrapper(x_train, y_train, x_test, y_test)

# ==================== 超参数 ====================
learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 2000

# ==================== 函数定义 ====================
def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1})
    correct_prediction = tf.equal(tf.argmax(y_pre, 1), tf.argmax(v_ys, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1})
    return result

def weight_variable(shape):
    initial = tf.random.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

# ==================== 构建网络 ====================
xs = tf.compat.v1.placeholder(tf.float32, [None, 784])   # 注意：数据已归一化，不再除以255
ys = tf.compat.v1.placeholder(tf.float32, [None, 10])
keep_prob = tf.compat.v1.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

# 卷积层 1
W_conv1 = weight_variable([7, 7, 1, 32])
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

# 全连接层 1
W_fc1 = weight_variable([7 * 7 * 64, 1024])
b_fc1 = bias_variable([1024])
h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# 全连接层 2
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

# 损失函数和优化器
cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.math.log(prediction),
                                              axis=1))
train_step = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

# ==================== 训练 ====================
with tf.compat.v1.Session() as sess:
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)

    for i in range(max_epoch):
        batch_xs, batch_ys = mnist.train.next_batch(100)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob: keep_prob_rate})
        if i % 100 == 0:
            acc = compute_accuracy(mnist.test.images[:1000], mnist.test.labels[:1000])
            print(f"Step {i}, Accuracy: {acc:.4f}")

Step 0, Accuracy: 0.0850
Step 100, Accuracy: 0.1260
Step 200, Accuracy: 0.1260
Step 300, Accuracy: 0.1260
Step 400, Accuracy: 0.1260
Step 500, Accuracy: 0.1260
Step 600, Accuracy: 0.1260
Step 700, Accuracy: 0.0940
Step 800, Accuracy: 0.0940
Step 900, Accuracy: 0.0940
Step 1000, Accuracy: 0.0940
Step 1100, Accuracy: 0.0940
Step 1200, Accuracy: 0.0940
Step 1300, Accuracy: 0.0940
Step 1400, Accuracy: 0.0940
Step 1500, Accuracy: 0.0940
Step 1600, Accuracy: 0.0940
Step 1700, Accuracy: 0.0940
Step 1800, Accuracy: 0.0940
Step 1900, Accuracy: 0.0940
